# 인용(Citations)

Claude API는 인용 기능을 제공하여, 문서에 대한 질문에 답할 때 Claude가 상세한 인용을 함께 제시할 수 있게 합니다. 인용은 응답에 담긴 정보의 출처를 사용자가 추적하고 검증할 수 있게 해 주는, 많은 LLM 기반 애플리케이션에서 유용한 기능입니다.

인용 기능을 지원하는 모델은 다음과 같습니다.
* `claude-sonnet-4-6`
* `claude-3-5-haiku-20241022`

인용 기능은 프롬프트 기반 인용 기법을 대체하는 방식입니다. 이 기능을 사용하면 다음과 같은 장점이 있습니다.
- 프롬프트 기반 기법에서는 인용하려는 원본 문서의 전체 인용문을 Claude가 그대로 출력해야 하는 경우가 많습니다. 그만큼 출력 토큰이 늘어나 비용도 증가합니다.
- 인용 기능은 유효한 출처로 제공되지 않은 문서나 위치를 가리키는 인용을 반환하지 않습니다.
- 테스트 결과, 인용 기능은 프롬프트 기반 기법보다 재현율(recall)과 정밀도(precision)가 더 높은 인용을 생성했습니다.

인용 기능에 대한 문서는 [여기](https://docs.claude.com/en/docs/build-with-claude/citations)에서 볼 수 있습니다.

## 준비

먼저 필요한 라이브러리를 설치하고 Anthropic 클라이언트를 초기화합니다.

In [ ]:
!pip install anthropic  --quiet

In [56]:
import json
import os

import anthropic

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
# ANTHROPIC_API_KEY = "" # Put your API key here!

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 문서 유형

인용 기능은 세 가지 문서 유형을 지원합니다. 출력되는 인용의 형식은 인용 대상 문서의 유형에 따라 달라집니다.

* 일반 텍스트 문서 인용 → 문자 위치(char location) 형식
* PDF 문서 인용 → 페이지 위치(page location) 형식
* 커스텀 콘텐츠 문서 인용 → 콘텐츠 블록 위치(content block location) 형식

아래 예제에서 각각을 어떻게 다루는지 살펴보겠습니다.

### 일반 텍스트 문서

일반 텍스트 문서 인용에서는 문서를 원시 텍스트 그대로 모델에 전달합니다. 문서는 하나만 줄 수도 있고 여러 개를 줄 수도 있습니다. 이 텍스트는 자동으로 문장 단위로 나뉘며, 모델은 적절한 문장을 인용합니다. 모델은 여러 문장을 한 번에 묶어 하나의 인용으로 만들 수 있지만, 문장보다 작은 단위는 인용하지 않습니다.

API 응답에는 출력된 텍스트와 함께 모든 인용에 대한 구조화된 데이터가 포함됩니다.

가상의 회사 PetWorld의 고객 지원 헬프센터 챗봇을 예로 들어 전체 예제를 살펴보겠습니다.

In [57]:
# Read all help center articles and create a list of documents
articles_dir = "./data/help_center_articles"
documents = []

for filename in sorted(os.listdir(articles_dir)):
    if filename.endswith(".txt"):
        with open(os.path.join(articles_dir, filename)) as f:
            content = f.read()
            # Split into title and body
            title_line, body = content.split("\n", 1)
            title = title_line.replace("title: ", "")
            documents.append(
                {
                    "type": "document",
                    "source": {"type": "text", "media_type": "text/plain", "data": body},
                    "title": title,
                    "citations": {"enabled": True},
                }
            )

QUESTION = "I just checked out, where is my order tracking number? Track package is not available on the website yet for my order."

# Add the question to the content
content = documents

response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    system="You are a customer support bot working for PetWorld. Your task is to provide short, helpful answers to user questions. Since you are in a chat interface avoid providing extra details. You will be given access to PetWorld's help center articles to help you answer questions.",
    messages=[
        {"role": "user", "content": documents},
        {
            "role": "user",
            "content": [{"type": "text", "text": f"Here is the user's question: {QUESTION}"}],
        },
    ],
)


def visualize_raw_response(response):
    raw_response = {"content": []}

    print("\n" + "=" * 80 + "\nRaw response:\n" + "=" * 80)

    for content in response.content:
        if content.type == "text":
            block = {"type": "text", "text": content.text}
            if hasattr(content, "citations") and content.citations:
                block["citations"] = []
                for citation in content.citations:
                    citation_dict = {
                        "type": citation.type,
                        "cited_text": citation.cited_text,
                        "document_title": citation.document_title,
                    }
                    if citation.type == "page_location":
                        citation_dict.update(
                            {
                                "start_page_number": citation.start_page_number,
                                "end_page_number": citation.end_page_number,
                            }
                        )
                    block["citations"].append(citation_dict)
            raw_response["content"].append(block)

    return json.dumps(raw_response, indent=2)


print(visualize_raw_response(response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Based on the documentation, I can explain why you don't see tracking yet: "
    },
    {
      "type": "text",
      "text": "You'll receive an email with your tracking number once your order ships. If you don't receive a tracking number within 48 hours of your order confirmation, please contact our customer support team for assistance.",
      "citations": [
        {
          "type": "char_location",
          "cited_text": "Once your order ships, you'll receive an email with a tracking number. ",
          "document_title": "Order Tracking Information"
        },
        {
          "type": "char_location",
          "cited_text": "If you haven't received a tracking number within 48 hours of your order confirmation, please contact our customer support team.",
          "document_title": "Order Tracking Information"
        }
      ]
    },
    {
      "type": "text",
      "text": "\n\nSince you just checked

#### 인용 시각화하기
인용 데이터를 활용하면 다음과 같은 UI를 만들 수 있습니다.

1. 정보의 출처가 정확히 어디인지 사용자에게 보여 주기
2. 원본 문서로 바로 연결하기
3. 인용된 텍스트를 맥락 속에서 강조 표시하기
4. 투명한 출처 표기로 신뢰 쌓기

아래는 Claude의 구조화된 인용 데이터를, 학술 논문처럼 번호가 매겨진 참고 문헌 형태의 읽기 쉬운 형식으로 바꿔 주는 간단한 시각화 함수입니다.

이 함수는 Claude의 응답 객체를 받아 다음을 출력합니다.
- 번호가 매겨진 인용 표시가 붙은 텍스트(예: "The answer [1] includes this fact [2]")
- 인용된 텍스트와 그 출처 문서를 보여 주는 번호 매긴 참고 문헌 목록

In [58]:
def visualize_citations(response):
    """
    Takes a response object and returns a string with numbered citations.
    Example output: "here is the plain text answer [1][2] here is some more text [3]"
    with a list of citations below.
    """
    # Dictionary to store unique citations
    citations_dict = {}
    citation_counter = 1

    # Final formatted text
    formatted_text = ""
    citations_list = []

    print("\n" + "=" * 80 + "\nFormatted response:\n" + "=" * 80)

    for content in response.content:
        if content.type == "text":
            text = content.text
            if hasattr(content, "citations") and content.citations:
                # Sort citations by their appearance in the text
                def get_sort_key(citation):
                    if hasattr(citation, "start_char_index"):
                        return citation.start_char_index
                    elif hasattr(citation, "start_page_number"):
                        return citation.start_page_number
                    elif hasattr(citation, "start_block_index"):
                        return citation.start_block_index
                    return 0  # fallback

                sorted_citations = sorted(content.citations, key=get_sort_key)

                # Process each citation
                for citation in sorted_citations:
                    doc_title = citation.document_title
                    cited_text = citation.cited_text.replace("\n", " ").replace("\r", " ")
                    # Remove any multiple spaces that might have been created
                    cited_text = " ".join(cited_text.split())

                    # Create a unique key for this citation
                    citation_key = f"{doc_title}:{cited_text}"

                    # If this is a new citation, add it to our dictionary
                    if citation_key not in citations_dict:
                        citations_dict[citation_key] = citation_counter
                        citations_list.append(
                            f'[{citation_counter}] "{cited_text}" found in "{doc_title}"'
                        )
                        citation_counter += 1

                    # Add the citation number to the text
                    citation_num = citations_dict[citation_key]
                    text += f" [{citation_num}]"

            formatted_text += text

    # Combine the formatted text with the citations list
    final_output = formatted_text + "\n\n" + "\n".join(citations_list)
    return final_output


formatted_response = visualize_citations(response)
print(formatted_response)


Formatted response:
Based on the documentation, I can explain why you don't see tracking yet: You'll receive an email with your tracking number once your order ships. If you don't receive a tracking number within 48 hours of your order confirmation, please contact our customer support team for assistance. [1] [2]

Since you just checked out, your order likely hasn't shipped yet. Once it ships, you'll receive the tracking information via email.

[1] "Once your order ships, you'll receive an email with a tracking number." found in "Order Tracking Information"
[2] "If you haven't received a tracking number within 48 hours of your order confirmation, please contact our customer support team." found in "Order Tracking Information"


### PDF 문서

PDF를 다룰 때 Claude는 특정 페이지 번호를 가리키는 인용을 제공할 수 있어 정보 출처를 추적하기 쉽습니다. PDF 인용은 다음과 같이 동작합니다.

- PDF 문서 내용은 base64로 인코딩된 데이터로 전달합니다
- 텍스트는 자동으로 문장 단위로 나뉩니다
- 인용에는 해당 정보가 있는 페이지 번호(1부터 시작)가 포함됩니다
- 모델은 여러 문장을 묶어 하나의 인용으로 만들 수 있지만, 문장보다 작은 단위는 인용하지 않습니다
- 이미지도 처리되기는 하지만, 현재로서는 텍스트 내용만 인용할 수 있습니다

아래는 Constitutional AI 논문으로 PDF 인용을 시연하는 예제입니다:

In [59]:
import base64
import json

# Read and encode the PDF
pdf_path = "data/Constitutional AI.pdf"
with open(pdf_path, "rb") as f:
    pdf_data = base64.b64encode(f.read()).decode()

pdf_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_data},
                    "title": "Constitutional AI Paper",
                    "citations": {"enabled": True},
                },
                {"type": "text", "text": "What is the main idea of Constitutional AI?"},
            ],
        }
    ],
)

print(visualize_raw_response(pdf_response))
print(visualize_citations(pdf_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Based on the paper, here are the key aspects of Constitutional AI:\n\n"
    },
    {
      "type": "text",
      "text": "Constitutional AI is a method for training a harmless AI assistant through self-improvement, without any human labels identifying harmful outputs. The only human oversight is provided through a list of rules or principles, hence the name \"Constitutional AI\".",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "We experiment with methods for training a harmless AI assistant through self\u0002improvement, without any human labels identifying harmful outputs. The only human\r\noversight is provided through a list of rules or principles, and so we refer to the method as\r\n\u2018Constitutional AI\u2019. ",
          "document_title": "Constitutional AI Paper",
          "start_page_number": 1,
          "end_page_number": 2
        }
      ]
    },
    {


### 커스텀 콘텐츠 문서

일반 텍스트 문서는 자동으로 문장 단위로 나뉘지만, 커스텀 콘텐츠 문서를 사용하면 인용 단위를 완전히 직접 제어할 수 있습니다. 이 API 형태로는 다음이 가능합니다.

* 원하는 크기로 청크를 직접 정의
* 최소 인용 단위 제어
* 문장 단위 분할이 잘 맞지 않는 문서에 맞게 최적화

아래 예제에서는 위의 일반 텍스트 예제와 동일한 헬프센터 문서를 사용하되, 문장 단위 인용을 허용하는 대신 각 문서를 하나의 청크로 취급합니다. 문서 유형 선택이 인용 동작과 세분성에 어떤 영향을 주는지 확인할 수 있습니다. `cited_text`가 원본 문서의 한 문장이 아니라 문서 전체가 되는 것을 볼 수 있습니다.

In [60]:
# Read all help center articles and create a list of custom content documents
articles_dir = "./data/help_center_articles"
documents = []

for filename in sorted(os.listdir(articles_dir)):
    if filename.endswith(".txt"):
        with open(os.path.join(articles_dir, filename)) as f:
            content = f.read()
            # Split into title and body
            title_line, body = content.split("\n", 1)
            title = title_line.replace("title: ", "")

            documents.append(
                {
                    "type": "document",
                    "source": {"type": "content", "content": [{"type": "text", "text": body}]},
                    "title": title,
                    "citations": {"enabled": True},
                }
            )

QUESTION = "I just checked out, where is my order tracking number? Track package is not available on the website yet for my order."

custom_content_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    system="You are a customer support bot working for PetWorld. Your task is to provide short, helpful answers to user questions. Since you are in a chat interface avoid providing extra details. You will be given access to PetWorld's help center articles to help you answer questions.",
    messages=[
        {"role": "user", "content": documents},
        {
            "role": "user",
            "content": [{"type": "text", "text": f"Here is the user's question: {QUESTION}"}],
        },
    ],
)

print(visualize_raw_response(custom_content_response))
print(visualize_citations(custom_content_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "You should receive an email with your tracking number once your order ships. If it's been less than 48 hours since your order confirmation, please wait as the tracking number may not be available yet. If you haven't received a tracking number after 48 hours, please contact our customer support team for assistance.",
      "citations": [
        {
          "type": "content_block_location",
          "cited_text": "Once your order ships, you'll receive an email with a tracking number. To track your package, log in to your PetWorld account and go to \"Order History.\" Click on the order you want to track and select \"Track Package.\" This will show you the current status and estimated delivery date. You can also enter the tracking number directly on our shipping partner's website for more detailed information. If you haven't received a tracking number within 48 hours of your order confirmation, please contact our 

### context 필드 사용하기

`context` 필드에는 Claude가 응답을 생성할 때 참고할 수 있지만 인용 대상은 되지 않는 문서 부가 정보를 담을 수 있습니다. 다음과 같은 경우에 유용합니다.

* 문서에 대한 메타데이터 제공(예: 발행일, 저자)
* [맥락 기반 검색(contextual retrieval)](https://www.anthropic.com/news/contextual-retrieval)
* 직접 인용되어서는 안 되는 사용 지침이나 맥락 포함

아래 예제에서는 멤버십 프로그램 문서를 제공하면서 `context` 필드에 주의 사항을 넣습니다. Claude가 context의 정보를 응답에 반영하면서도, context 필드의 내용은 인용 대상이 되지 않는다는 점에 주목하세요.

In [61]:
import json

# Create a document with context field
document = {
    "type": "document",
    "source": {
        "type": "text",
        "media_type": "text/plain",
        "data": "PetWorld offers a loyalty program where customers earn 1 point for every dollar spent. Once you accumulate 100 points, you'll receive a $5 reward that can be used on your next purchase. Points expire 12 months after they are earned. You can check your point balance in your account dashboard or by asking customer service.",
    },
    "title": "Loyalty Program Details",
    "context": "WARNING: This article has not been updated in 12 months. Content may be out of date. Be sure to inform the user this content may be incorrect after providing guidance.",
    "citations": {"enabled": True},
}

QUESTION = "How does PetWorld's loyalty program work? When do points expire?"

context_response = client.messages.create(
    model="claude-sonnet-4-6",
    temperature=0.0,
    max_tokens=1024,
    messages=[{"role": "user", "content": [document, {"type": "text", "text": QUESTION}]}],
)

print(visualize_raw_response(context_response))
print(visualize_citations(context_response))


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "Let me explain PetWorld's loyalty program based on the provided information:\n\n"
    },
    {
      "type": "text",
      "text": "PetWorld's loyalty program is straightforward - you earn 1 point for every dollar you spend. These points can be redeemed once you reach 100 points, which will get you a $5 reward that you can use on your next purchase.",
      "citations": [
        {
          "type": "char_location",
          "cited_text": "PetWorld offers a loyalty program where customers earn 1 point for every dollar spent. Once you accumulate 100 points, you'll receive a $5 reward that can be used on your next purchase. ",
          "document_title": "Loyalty Program Details"
        }
      ]
    },
    {
      "type": "text",
      "text": "\n\n"
    },
    {
      "type": "text",
      "text": "Points have an expiration period of 12 months from the date they are earned.",
      "citations": [
        {
   

### PDF 하이라이팅

PDF 인용의 한 가지 한계는 페이지 번호만 반환된다는 점입니다. 서드파티 라이브러리를 사용해 반환된 인용 텍스트를 페이지 내용과 대조하면 인용된 부분을 눈에 띄게 표시할 수 있습니다. 이 셀에서는 Claude와 PyMuPDF를 사용해 PDF 인용을 하이라이팅하고 주석이 달린 새 PDF를 만드는 방법을 보여 줍니다:

In [62]:
import fitz  # PyMuPDF

# Setup paths and read PDF
pdf_path = "data/Amazon-com-Inc-2023-Shareholder-Letter.pdf"
output_pdf_path = "data/Amazon-com-Inc-2023-Shareholder-Letter-highlighted.pdf"

# Read and encode the PDF
with open(pdf_path, "rb") as f:
    pdf_data = base64.b64encode(f.read()).decode()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_data},
                    "title": "Amazon 2023 Shareholder Letter",
                    "citations": {"enabled": True},
                },
                {
                    "type": "text",
                    "text": "What was Amazon's total revenue in 2023 and how much did it grow year-over-year?",
                },
            ],
        }
    ],
)

print(visualize_raw_response(response))

# Collect PDF citations
pdf_citations = []
for content in response.content:
    if hasattr(content, "citations") and content.citations:
        for citation in content.citations:
            if citation.type == "page_location":
                pdf_citations.append(citation)

doc = fitz.open(pdf_path)

# Process each citation
for citation in pdf_citations:
    if citation.type == "page_location":
        text_to_find = citation.cited_text.replace("\u0002", "")
        start_page = citation.start_page_number - 1  # Convert to 0-based index
        end_page = citation.end_page_number - 2

        # Process each page in the citation range
        for page_num in range(start_page, end_page + 1):
            page = doc[page_num]

            text_instances = page.search_for(text_to_find.strip())

            if text_instances:
                print(f"Found cited text on page {page_num + 1}")
                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.set_colors({"stroke": (1, 1, 0)})  # Yellow highlight
                    highlight.update()
            else:
                print(f"{text_to_find} not found on page {page_num + 1}")

# Save the new PDF
doc.save(output_pdf_path)
doc.close()

print(f"\nCreated highlighted PDF at: {output_pdf_path}")


Raw response:
{
  "content": [
    {
      "type": "text",
      "text": "According to the letter, "
    },
    {
      "type": "text",
      "text": "Amazon's total revenue grew 12% year-over-year (\"YoY\") from $514B to $575B in 2023",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "In 2023, Amazon\u2019s total revenue grew 12% year-over-year (\u201cYoY\u201d) from $514B to $575B. ",
          "document_title": "Amazon 2023 Shareholder Letter",
          "start_page_number": 1,
          "end_page_number": 2
        }
      ]
    },
    {
      "type": "text",
      "text": ".\n\nBreaking this down by segment:\n"
    },
    {
      "type": "text",
      "text": "\n- North America revenue increased 12% YoY from $316B to $353B\n- International revenue grew 11% YoY from $118B to $131B  \n- AWS revenue increased 13% YoY from $80B to $91B",
      "citations": [
        {
          "type": "page_location",
          "cited_text": "By segment, Nor